# Deep Learning for Business Applications course

## TOPIC 2: Introduction to Computer Vision. Intro to CNNs

*Notebook for MBA (Master of Business Analytics) students. No deep math required — only intuition, pictures, and running the code.*

Based on the [MNIST dataset](https://github.com/DeepTrackAI/MNIST_dataset) and classic PyTorch tutorials.

## 0. The big picture: what are we actually doing?

A **digital image is just a grid of numbers**: each pixel is a brightness from 0 (black) to 255 (white). Recognizing a digit means finding *patterns of numbers* — a "7" is a specific shape of dark pixels with a diagonal stroke.

- **Classical way** (next notebooks with OpenCV): a human writes the rules by hand — "an edge is where brightness changes sharply", etc.
- **Neural network way** (this notebook): we show the machine thousands of labeled examples, and it *learns* the rules itself.

We will do it in two steps, from simple to powerful:
1. **Logistic regression** — one linear layer: like judging each pixel independently ("is this pixel dark? then it's probably a 1"). Fast, but ignores *where* pixels are relative to each other.
2. **CNN (convolutional network)** — small "stamps" (filters) slide over the image looking for local shapes: strokes, corners, loops. This sees *structure*, and scores much higher.

Everyday analogy: logistic regression is checking a photo with a straw — you see isolated dots; a CNN looks at the whole picture.

## 1. Libraries

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## 2. MNIST dataset

About MNIST dataset:
- handwritten digits 0, 1, …, 9
- 28x28 size (784 pixels)
- training set of 60 000 and a test set of 10 000 examples
- grey scale colored

#### 2.1. Load the dataset

In [ ]:
train_data = torchvision.datasets.MNIST('./data', download=True)
test_data = torchvision.datasets.MNIST('data', train=False)
print(train_data)
print(test_data)

In [ ]:
plt.subplot(1, 2, 1)
image, label = train_data[0]
plt.imshow(image, cmap='gray')
plt.title('label of image: {}'.format(label), fontsize=14)
plt.subplot(1, 2, 2)
image, label = train_data[1]
plt.imshow(image, cmap='gray')
plt.title('label of image: {}'.format(label), fontsize=14)
plt.show()

In [ ]:
type(image)

In [ ]:
img_arr = np.asarray(image)
img_arr.shape

In [ ]:
img_arr

#### 2.2. Basic transformations

In [ ]:
# NOTE: NN likes input in range (0, 1)
# Convert input images to tensors and normalize
# Usually, 'transforms.ToTensor()' is used to turn
# the input data in the range # of [0,255]
# to a 3-dimensional Tensor.
# This function automatically scales the
# input data to the range of [0,1].
# (This is equivalent to scaling the data down to 0,1)
# The scaled mean and standard deviation
# of the MNIST dataset (precalculated):
# data_mean = 0.1307
# data_std = 0.3081

transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    #torchvision.transforms.Normalize(  # apply if needed
    #    (data_mean, ),
    #    (data_std, )
    #)
    ])
train_data = torchvision.datasets.MNIST(
    'data',
    train=True,
    transform=transform
)
test_data = torchvision.datasets.MNIST(
    'data',
    train=False,
    transform=transform
)

## 3. Models

Training a network on 60 000 images cannot be done all at once — you feed them in **batches** (here: 100 images at a time), like serving dinner in portions instead of one giant plate.

- `DataLoader` — the "conveyor belt": it takes the dataset, shuffles it, and serves ready batches.
- Shuffling the training set (`shuffle=True`) prevents the network from memorizing the *order* of examples.

### 3.1. Data loaders for training pipeline

In [ ]:
BATCH_SIZE = 100
INPUT_SIZE = 28 * 28
NUM_CLASSES = 10
LEARNING_RATE = .001
NUM_EPOCHS = 5

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

The simplest possible classifier: one fully connected layer. Every pixel gets its own weight ("how much does darkness here point to digit X?"), we sum everything and get 10 scores — one per digit. The highest score wins.Read the training loop below as a flashcard routine:
| Code | Simple meaning |
|:-----|:---------------|
| `output = model(images)` | guess the answer |
| `compute_loss = loss(...)` | grade the guess ("how wrong was it?") |
| `compute_loss.backward()` | backpropagation (Topic 1!): distribute the blame to every weight |
| `optimizer.step()` | turn every knob a little in the right direction |
| `optimizer.zero_grad()` | wipe the whiteboard clean before the next example |

Cross-entropy loss is just the "grading rubric" for classification: it punishes confident wrong answers heavily and mild mistakes lightly.

### 3.2. Logistic regression

Let's implement logistic regression with help of Pytorch:

In [ ]:
# Just one fully connected layer
# will give us a regression

class LogisticRegression(nn.Module):
    def __init__(self, input_size, num_classes):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, feature):
        output = self.linear(feature)
        return output

In [ ]:
model = LogisticRegression(
    INPUT_SIZE,
    NUM_CLASSES
)

# The Cross-Entropy Loss is derived from the principles of maximum
# likelihood estimation when applied to the task of classification.
# Maximizing the likelihood is equivalent to minimizing the negative
# log-likelihood. In classification, the likelihood function can be
# expressed as the product of the probabilities of the correct classes:
# Binary Cross-Entropy Loss and Multiclass Cross-Entropy Loss
# are two variants of cross-entropy loss, each tailored to different
# types of classification tasks:
loss = nn.CrossEntropyLoss()

# about `Adam` optimizer
# https://pytorch.org/docs/stable/generated/torch.optim.Adam.html
# it works well in most of cases
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
for epoch in range(NUM_EPOCHS):
    correct = 0
    for i, (images, labels) in enumerate(train_loader):
        images = torch.autograd.Variable(images.view(-1, INPUT_SIZE))
        labels = torch.autograd.Variable(labels)

        # nullify gradients w.r.t. parameters
        optimizer.zero_grad()
        # forward propagation
        output = model(images)
        # compute loss based on obtained value and actual label
        compute_loss = loss(output, labels)
        # backward propagation
        compute_loss.backward()
        # update the parameters
        optimizer.step()

        # Total correct predictions
        predicted = torch.max(output.data, 1)[1]
        correct += (predicted == labels).sum()
        if i % 50 == 0:
            print(
                'Epoch {} - training [{}/{} ({:.0f}%)] loss: {:.3f}, accuracy: {:.2f}%'.format(
                    epoch,
                    i * len(images),
                    len(train_loader.dataset),
                    100 * i / len(train_loader),
                    compute_loss.item(),
                    float(correct * 100) / float(BATCH_SIZE * (i + 1))
                ),
                end='\r'
            )

    # check total accuracy of predicted value and actual label
    accurate = 0
    total = 0
    for images, labels in test_loader:
        images = torch.autograd.Variable(images.view(-1, INPUT_SIZE))
        output = model(images)
        _, predicted = torch.max(output.data, 1)
        compute_loss = loss(output, labels)
        # total labels
        total += labels.size(0)

        # Total correct predictions
        accurate += (predicted == labels).sum()
        accuracy_score = 100 * accurate/total

    print('Epoch {} - validation loss: {:.3f}, validation accuracy: {:.2f}%        '.format(
        epoch,
        compute_loss.item(),
        accuracy_score
    ))

Now let's compare: same task, but with a **convolutional neural network**.

Few points before network building:
- [about](https://docs.python.org/3/library/functions.html#super) `super` in Python. It allows you to call methods defined in the superclass from the subclass, enabling you to extend and customize the functionality inherited from the parent class
- [about](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) `Conv2d` layer in Pytorch
- [about](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) `Dropout` layers in Pytorch

The three new ingredients, in plain words:

- **Conv2d** — a small stamp (3x3 pixels) slides across the whole image and fires when it matches a local pattern (a stroke, a corner). Early layers find simple shapes, deeper layers combine them ("loop + line = 9").
- **max_pool2d** — zoom-out: keep only the strongest activation from each 2x2 patch. The network cares *that* a pattern was found, not exactly where.
- **Dropout** — during training, randomly switch off half of the neurons. Like studying with some answers covered: forces the network not to rely on a single neuron, so it generalizes better (helps fight overfitting).

### 3.3. CNN approach

Now will use simple convolutional neural network with couple of convolutional layers. Few points before network building:
- [about](https://docs.python.org/3/library/functions.html#super) `super` in Python. It allows you to call methods defined in the superclass from the subclass, enabling you to extend and customize the functionality inherited from the parent class
- [about](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) `Conv2d` layer in Pytorch
- [about](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) `Dropout` layers in Pytorch

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding='valid')
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding='valid')
        self.dropout1 = nn.Dropout(.5)
        self.dropout2 = nn.Dropout(.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)  # you may need to comment this line for HA
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)  # you may need to comment this line for HA
        x = self.fc2(x)
        output = F.softmax(x, dim=1)
        return output

In [ ]:
model = SimpleCNN(NUM_CLASSES)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)
loss = nn.CrossEntropyLoss()

So logistic regression reaches ~92% accuracy; the CNN should reach ~98%+ — that gap is what "seeing structure" is worth.

### How did we know the size 9216?

Each `Conv2d` with `kernel_size=3` and no padding shrinks a 28x28 image by 2 pixels per side: 28 → 26 → 24, then max-pooling halves it: 24 → 12. With 64 channels that is 64 × 12 × 12 = **9216** numbers, flattened before the dense layers. You rarely compute this by hand in practice — print `model` and check `torch.flatten` output shapes instead.

## HOME ASSIGNMENT

You have to make few experiments:
1. Just run training CNN framework (code cell below) and observe if validation accuracy changes (get higher or lower)
2. Try to play with `Dropout` layers e.g. remove them, re-run model creation and training cells of the notebook, observe if validation accuracy changes (get higher or lower)
3. __(ADVANCED)__ Try CNN without `max_pool2d` layer, what has happened?

In [ ]:
for epoch in range(NUM_EPOCHS):
    correct = 0
    for i, (images, labels) in enumerate(train_loader):
        images = torch.autograd.Variable(images)
        labels = torch.autograd.Variable(labels)

        # Nullify gradients w.r.t. parameters
        optimizer.zero_grad()
        # forward propagation
        output = model(images)
        # compute loss based on obtained value and actual label
        compute_loss = loss(output, labels)
        # backward propagation
        compute_loss.backward()
        # update the parameters
        optimizer.step()

        # total correct predictions
        predicted = torch.max(output.data, 1)[1]
        correct += (predicted == labels).sum()
        if i % 50 == 0:
            print(
                'Epoch {} - training [{}/{} ({:.0f}%)] loss: {:.3f}, accuracy: {:.2f}%'.format(
                    epoch,
                    i * len(images),
                    len(train_loader.dataset),
                    100 * i / len(train_loader),
                    compute_loss.item(),
                    float(correct * 100) / float(BATCH_SIZE * (i + 1))
                ),
                end='\r'
            )

    # check total accuracy of predicted value and actual label
    accurate = 0
    total = 0
    for images, labels in test_loader:
        images = torch.autograd.Variable(images)
        output = model(images)
        _, predicted = torch.max(output.data, 1)
        compute_loss = loss(output, labels)
        # total labels
        total += labels.size(0)

        # total correct predictions
        accurate += (predicted == labels).sum()
        accuracy_score = 100 * accurate/total

    print('Epoch {} - validation loss: {:.3f}, validation accuracy: {:.2f}%        '.format(
        epoch,
        compute_loss.item(),
        accuracy_score
    ))

## Additional questions to think about (no code needed)

1. Logistic regression flattens the image first (`images.view(-1, INPUT_SIZE)`), the CNN does not. Why does the CNN *need* the 2D grid preserved?
2. Which digit pairs do you expect the model to confuse most (look at its errors)? Why are they business-relevant (e.g. hand-written amounts on invoices)?
3. Dropout is ON during training but OFF during evaluation automatically. Why would keeping it ON at prediction time be a bad idea?